# GLM
The goal of the GLM is to investigate the activation of the brain in the math and story tasks.

a) Build an appropriate experimental design matrix that you will use in the following GLM analysis.

b) Compute and plot the correlation matrix between the regressors of the design matrix and comment on the results.

c) Run the GLM at the level you deem reasonable and report the statistical maps of each of the task-related regressors. Briefly comment on the statistical maps.

d) Using the results of the GLM analysis and a contrast vector of your choice, create the activation maps corresponding to the story vs math
contrast. Please include in the report the contrast vector used and the contrast map you obtained.

e) Discuss which brain regions and networks are more activated during numerical than semantic processing, and which are more activated in the opposite contrast. As brain activity at rest can be separated into known functional networks [2], comment the results in terms of functional networks. Report known functional networks found within the statistical results and brain activation maps. You can either observe their overlaps or use brain atlas to inform your answer.

f) [Hard] Based on the literature, some regions should stand out as particularly relevant to semantic language processing. Are these regions present in your contrast maps? If not, discuss at least two possible reasons that could explain their absence.

## Loading data
The GLM needs one time series per voxel, since the preprocessed data is already 4D:

In [1]:
import os

import nibabel as nib
import pandas as pd

# Definig the TR of the protocol
tr = ??

fmri_img = nib.load(path_to_preprocessed_4d)

SyntaxError: invalid syntax (378071879.py, line 7)

Regarding the events, Nilearn needs a DataFrame with exactly three columns: onset (s), duration (s) and trial_type. Each unique trial_type becomes one regressor. 

In [5]:
# Defining the path of the events
events_path = os.path.join(data_dir, 'fMRI', 'tfMRI_LANGUAGE_LR', 'tfMRI_LANGUAGE_LR_task_onsets.csv')

# Reading and printing the events
events = pd.read_csv(events_path)
events

,onset,duration,trial_type
0,0.241,28.270,story
1,0.241,20.445,present_story
2,21.192,1.161,question_story
3,25.550,2.961,response_story
4,28.696,0.727,cue
5,30.095,5.667,present_math
6,30.095,12.241,math
7,35.946,3.391,question_math
8,39.371,2.965,response_math
9,42.570,5.453,present_math


## 3a. Building the design matrix
In order to build the GLM, first of all we need to specify EXPLAIN

We decided to add the following regressors: 
- Signal drift (slow scanner or physiological trends, modelled as low-order polynomials or cosines)
- Censoring (explain what it is)
- Motion parameters

In [ ]:
from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix

# Defining the regressors: motion parameters
mot_params = load_mot_params_fsl_6_dof(path_moco_data + ".par")
assert len(mot_params) == n_scans   # chek there is exacly one line per volume

regressors = mot_params.copy()

# Defining the regressors: censoring
outlier_vols = [Indici dei volumi above threshold, 24, 25, 44]
for i, v in enumerate(outlier_vols):
    reg = np.zeros((n_scans,))
    reg[v] = 1.0
    regressors['motion outlier#' + str(i+1)] = reg


design_mat = fmri_glm.design_matrices_[0]         # start from the task design matrix (a pandas DataFrame)
for (col, name) in zip(column_values.T, column_names):
design_mat[name] = col                        # adding a regressor = adding a column


design_mat = make_first_level_design_matrix(frame_times,
                                            events,
                                            hrf_model='spm',
                                            drift_model='cosine',
                                            high_pass=.01,
                                            add_regs=confounds.values,
                                            add_reg_names=list(confounds.columns))

plot_design_matrix(design_mat)
plt.title('Design matrix')
plt.show()

## 3b. Correlation between regressors


In [ ]:
cols = [c for c in design_mat_regged.columns if c != 'constant'] # the constant has zero variance → NaN correlations 
corr = design_mat_regged[cols].corr()

plt.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r'); plt.colorbar() plt.xticks(range(len(cols)), cols, rotation=90); plt.yticks(range(len(cols)), cols) plt.show() 

# Per il plot Claude mi ha suggerito: 
plt.figure(figsize=(8, 8))
plt.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='Pearson r')
plt.xticks(range(len(corr)), corr.columns, rotation=90)
plt.yticks(range(len(corr)), corr.columns)
plt.title('Correlation between regressors')
plt.show()

## 3c. Fitting the model
Since there is only one subject, we need to condutc a first level analysis. 

In [ ]:
from nilearn.glm.first_level import FirstLevelModel

# DA DECIDERE CHE TIPI DI DRIFTING REGRESSORS SONO IDEALI DA UTILIZZARE QUA, IL LAB USA POLINOMIALI

#from nilearn.glm.first_level.design_matrix import _make_drift
#column_values, column_names = _make_drift(drift_model='polynomial',
#                           frame_times=np.asarray(list(range(0, 666, 7))),  # 96 scans × TR 7 s
#                                          order=3, high_pass=None)


# Define what type of GLM we want (nature of the noise, repetition time of the data...)
# The function creates an empty model object that still needs to be fitted
# DA CAPIRE QUALI PARAMETRI SI TENGONO E QUALI NO
fmri_glm = FirstLevelModel(t_r=tr,          # repetition time: tells nilearn when each scan was acquired
			         noise_model='ar1',     # AR(1) pre-whitening for autocorrelated noise
			         standardize=False,     # keep signal units (betas in original scale)
			         hrf_model='spm',       # canonical double-gamma HRF used for convolution
			         drift_model='cosine',  # automatically adding cosine drft regressors
			         high_pass=.01)         # cutoff (Hz) — only used by the 'cosine' drift model, so ignored here

# Cancellare righe sopra??
fit_glm = FirstLevelModel(t_r=7.0, drift_model='None', minimize_memory=False) 
fit_glm.fit(fmri_img, design_matrices=[design_mat])

# Manca la statistical maps of each of the task-related regressors.

## 3d. Contrast and activation maps activation "story vs math"